# nanomoe: quickstart

A dense transformer sends every token through the same feed-forward network. A **Mixture-of-Experts** (MoE) model
keeps several smaller networks (experts) and a **router** that sends each token to the top 2 of them, plus one
**shared** expert every token uses. More weights in total, the same work per token.

This notebook runs the tests, trains a dense model and an MoE with the **same active parameters** on Tiny
Shakespeare, and draws which kinds of characters go to which expert. On a Colab GPU the default settings take a few
minutes; on a CPU, lower `STEPS`.

In [ ]:
# Setup: use this checkout if we're inside it, otherwise clone it (torch is preinstalled on Colab).
import os, subprocess, sys
if os.path.exists("../nanomoe.py"):
    os.chdir("..")
elif not os.path.exists("nanomoe.py"):
    subprocess.run(["git", "clone", "-q", "--depth", "1",
                    "https://github.com/albertlieyingadrian/nanomoe-transformer"], check=True)
    os.chdir("nanomoe-transformer")
import importlib.util
missing = [m for m in ("matplotlib", "pytest") if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
STEPS = int(os.environ.get("NANOMOE_STEPS", 1500))   # training steps per model
print(os.getcwd(), "steps:", STEPS)

## 1. The tests

Shapes, matched active parameters, dispatch against a token-by-token reference, the load-balancing loss at its two extremes, bias balancing on a lopsided router, the MLA cache, and a short training run.

In [ ]:
!{sys.executable} -m pytest -q test_nanomoe.py

## 2. Dense vs MoE at the same active parameters

`dense` has one SwiGLU network as wide as 3 experts. `moe-bias` has 8 routed experts (top 2) and 1 shared expert,
balanced by DeepSeek-V3's auxiliary-loss-free bias. Each token uses the same number of weights in both; the MoE
stores about twice as many.

In [ ]:
!{sys.executable} nanomoe.py --mode compare --steps {STEPS} --variants dense,moe-bias --out out/nb.pt

In [ ]:
import json
import matplotlib.pyplot as plt
res = json.load(open("out/nb_compare.json"))["results"]
for name, r in res.items():
    steps, vals = zip(*r["curve"])
    plt.plot(steps, vals, "-o", ms=3, label=f"{name}: {r['active']:,} active / {r['params']:,} total")
plt.xlabel("step"); plt.ylabel("validation loss"); plt.legend(); plt.title("same active parameters"); plt.show()

## 3. Where do tokens go?

Rows are kinds of character, columns are experts; each cell is the share of that kind's routing slots that went to
that expert. The title gives the mutual information between the character and its top expert, as a fraction of the
expert's entropy (0 = routing ignores the character, 1 = the character decides). The printout adds two controls:
the same labels shuffled, and an untrained model.

In [ ]:
!{sys.executable} nanomoe.py --mode heatmap --out out/nb_moe-bias.pt
from IPython.display import Image, display
display(Image("out/nb_moe-bias_heatmap.png"))

In [ ]:
!{sys.executable} nanomoe.py --mode sample --out out/nb_moe-bias.pt

## Where to go next

- `program.md` lists experiments: `--balance aux` or `none`, `--router_skew 3` to provoke routing collapse,
  `--fine 4` for fine-grained experts, `--capacity_factor 1.0` to drop tokens, `--mtp 1` for multi-token prediction.
- The course this file comes from builds every piece in its own tested stage:
  https://github.com/albertlieyingadrian/build-a-moe-transformer